In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import re
import networkx as nx
import plotly.graph_objects as go

from typing import Any, Iterable, Optional

from IPython.display import display, Markdown
from dotenv import load_dotenv
from openai import OpenAI, BadRequestError
from pyvis.network import Network

In [41]:
# Causality graph build based on topn n feature importance

# # 2023-07-11 -> 2023-07-17, 
# # 2023-07-18 -> 2023-07-24, 
# # 2023-07-25 -> 2023-08-01, 
# # 2023-08-02 -> 2023-08-09



start_date = "2023-07-11"
end_date = "2023-07-17"
top_n = 20

feature_importance = pd.read_csv(f"../../feature_importance/feature_importance_{start_date}_{end_date}/feature_importance_from_{start_date}_to_{end_date}_top_{top_n}.csv")  
causality_graph = pd.read_csv(f"../../causality/causality_{start_date}_{end_date}/causality_from_{start_date}_to_{end_date}_top_{top_n}.csv")

In [42]:
feature_importance["Feature name"]

0                   Delivery
1       delivery_sales_ratio
2                 Production
3     production_sales_ratio
4              Factory_Issue
5            inventory_proxy
6               Sales_lag_21
7                Sales_lag_1
8                Sales_lag_2
9               Sales_lag_28
10               Sales_lag_7
11               Sales_lag_3
12              Sales_lag_14
13        factory_issue_rate
14        Product_SOS001L12P
15          balance_of_trade
16        stock_market_close
17              Sub-Group_SO
18                   exports
19               Temperature
Name: Feature name, dtype: str

In [43]:
causality_graph

,cause,effect,lag,coefficient,direction
0,Sales_lag_28,Production,0.0,0.932276,positive
1,delivery_sales_ratio,production_sales_ratio,0.0,0.909791,positive
2,Sales_lag_28,Sales_lag_3,0.0,0.850840,positive
3,Sales_lag_28,Sales_lag_7,0.0,0.834155,positive
4,Sales_lag_7,Sales_lag_21,0.0,0.747857,positive
...,...,...,...,...,...
103,Product_SOS001L12P,Delivery,1.0,-0.055638,negative
104,Sales_lag_14,Sales_lag_1,1.0,-0.055528,negative
105,Product_SOS001L12P,Factory_Issue,1.0,-0.055279,negative
106,Product_SOS001L12P,Sales,1.0,-0.050122,negative


In [44]:
macroeconomic_features = ['interest_rate_value', 'stock_market_close', 'inflation_rate', 'balance_of_trade', 'exports', 'imports', 'currency_log_return', 'currency_momentum_5d']
macroeconomic_features

['interest_rate_value',
 'stock_market_close',
 'inflation_rate',
 'balance_of_trade',
 'exports',
 'imports',
 'currency_log_return',
 'currency_momentum_5d']

In [45]:
# Check how many macroeconomic features exist in feature_importance["Feature name"]
existing_macro = [f for f in macroeconomic_features if f in feature_importance["Feature name"].values]
missing_macro = [f for f in macroeconomic_features if f not in feature_importance["Feature name"].values]

print(f"Found {len(existing_macro)} of {len(macroeconomic_features)} macroeconomic features in feature_importance:")
for f in existing_macro:
    print(f"  ✓ {f}")

if missing_macro:
    print(f"\nMissing {len(missing_macro)} features:")
    for f in missing_macro:
        print(f"  ✗ {f}")

Found 3 of 8 macroeconomic features in feature_importance:
  ✓ stock_market_close
  ✓ balance_of_trade
  ✓ exports

Missing 5 features:
  ✗ interest_rate_value
  ✗ inflation_rate
  ✗ imports
  ✗ currency_log_return
  ✗ currency_momentum_5d


In [46]:
causality_graph[
    causality_graph["cause"].isin(existing_macro)
    | causality_graph["effect"].isin(existing_macro)
]

,cause,effect,lag,coefficient,direction
8,Temperature,stock_market_close,0.0,0.590163,positive
9,Temperature,stock_market_close,1.0,-0.580860,negative
37,stock_market_close,Sales_lag_2,0.0,-0.186389,negative
76,production_sales_ratio,stock_market_close,1.0,-0.103674,negative
78,stock_market_close,Sales_lag_1,1.0,0.089111,positive
95,delivery_sales_ratio,stock_market_close,1.0,0.068625,positive
99,stock_market_close,Sales_lag_3,0.0,-0.063023,negative
107,exports,NaN,NaN,0.000000,none


In [7]:
from datetime import date, datetime
from pathlib import Path


def _to_date(value):
    """Parse 'YYYY-MM-DD' (or a date/datetime) into a date; return None if unusable."""
    if value is None:
        return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    text = str(value).strip()
    if not text:
        return None
    try:
        return datetime.fromisoformat(text[:10]).date()
    except ValueError:
        return None


def load_llm_article_connections(folder_path, limit_file="none", start_date=None, end_date=None, verbose=True):
    """Collect article metadata + llm_connections from the LLM response json files.

    folder_path : folder holding the llm_response_*.json files
    limit_file  : "none"/None -> every json file in the folder, otherwise a number ->
                  only the first N files (files sorted by name)
    start_date  : inclusive lower bound on article["date"], "YYYY-MM-DD" or None
    end_date    : inclusive upper bound on article["date"], "YYYY-MM-DD" or None

    Returns a list of {"article": {source, date, url, body_preview}, "llm_connections": [...]}.
    """
    folder = Path(folder_path)
    files = sorted(folder.glob("*.json"))

    if limit_file is not None and str(limit_file).strip().lower() != "none":
        files = files[: int(limit_file)]

    start = _to_date(start_date)
    end = _to_date(end_date)

    records = []
    out_of_range = 0
    unreadable = 0

    for path in files:
        try:
            with open(path, encoding="utf-8") as f:
                payload = json.load(f)
        except (OSError, json.JSONDecodeError) as exc:
            unreadable += 1
            if verbose:
                print(f"  skipped {path.name}: {exc}")
            continue

        article = payload.get("article") or {}
        article_date = _to_date(article.get("date"))

        if article_date is None:
            out_of_range += 1
            continue
        if start is not None and article_date < start:
            out_of_range += 1
            continue
        if end is not None and article_date > end:
            out_of_range += 1
            continue

        records.append(
            {
                "article": {
                    "source": article.get("source"),
                    "date": article.get("date"),
                    "url": article.get("url"),
                    "body_preview": article.get("body_preview"),
                },
                "llm_connections": payload.get("llm_connections", []),
            }
        )

    if verbose:
        n_conn = sum(len(r["llm_connections"]) for r in records)
        print(
            f"{len(files)} file(s) scanned in {folder} | kept {len(records)} article(s) "
            f"with {n_conn} llm_connection(s) | {out_of_range} outside date range or undated "
            f"| {unreadable} unreadable"
        )

    return records

In [ ]:
def unique_article(data, verbose=True):
    """Drop the records of `data` that describe an article already seen.

    data    : list returned by load_llm_article_connections()
    verbose : print how many records were collapsed

    Two records describe the same article when their "source", "date" and "url" all
    match (exact match, ignoring surrounding whitespace and letter case). The first
    record of each article is kept, in the original order.

    Returns the unique records (same shape as `data`).
    """
    keys = ("source", "date", "url")

    seen = set()
    unique = []

    for record in data:
        article = record.get("article") or {}
        identity = tuple(str(article.get(key) or "").strip().casefold() for key in keys)

        if identity in seen:
            continue

        seen.add(identity)
        unique.append(record)

    if verbose:
        print(
            f"{len(data)} record(s) in | kept {len(unique)} unique article(s) "
            f"| {len(data) - len(unique)} duplicate(s) dropped"
        )

    return unique

In [20]:
def find_article(data, search_type, value):
    """Find the records of `data` whose article field `search_type` equals `value`.

    data        : list returned by load_llm_article_connections()
    search_type : "source", "date" or "url"
    value       : the value to match on that field (exact match, ignoring surrounding
                  whitespace and letter case)

    Returns the matching records (same shape as `data`), empty list if nothing matches.
    """
    allowed = ("source", "date", "url")
    if search_type not in allowed:
        raise ValueError(f"search_type must be one of {allowed}, got {search_type!r}")

    wanted = str(value).strip().casefold()

    return [
        record
        for record in data
        if str((record.get("article") or {}).get(search_type) or "").strip().casefold() == wanted
    ]

In [21]:
event_graph_json_dir = "../../event_graph/all_articles/json"

# limit_file="none" -> every json file; limit_file=10 -> only the first 10 files
articles = load_llm_article_connections(
    event_graph_json_dir,
    limit_file="none",
    start_date="2022-01-01",
    end_date=end_date,
)

unique_articles = unique_article(articles)

736 file(s) scanned in ..\..\event_graph\all_articles\json | kept 721 article(s) with 906 llm_connection(s) | 15 outside date range or undated | 0 unreadable
721 record(s) in | kept 381 unique article(s) | 340 duplicate(s) dropped


In [23]:
find_article(unique_articles, "source", "AP NEWS")


[{'article': {'source': 'AP NEWS',
   'date': '2022-09-23',
   'url': 'https://apnews.com/article/health-bangladesh-covid-global-trade-33c74e143358d10db75eb3511e385732',
   'body_preview': 'DHAKA, Bangladesh (AP) -- Standing in line to try to buy food, Rekha Begum is distraught. Like many others in Bangladesh, she is struggling to find affordable daily essentials like rice, lentils and onions.\n\n"I went to two other places, but they told me they don\'t have supplies. Then I came here and stood at the end of the queue," said Begum, 60, as she waited for nearly two hours to buy what she needed from a truck selling food at subsidized prices in the capital, Dhaka.\n\nBangladesh\'s economic '},
  'llm_connections': [{'feature': 'inflation_rate',
    'event_summary': 'Ukraine war raised commodity prices',
    'edge_label': 'raises price levels',
    'evidence_quote': 'The war in Ukraine has pushed higher prices of many commodities at a time when they already were surging',
    'scope_justif

<br> <br>

## Merging Causal Graph and Event Graph

In [24]:
MERGED_COLUMNS = [
    # shared edge description
    "edge_type", "source_node", "source_type", "target_node", "target_type", "edge_label",
    # the join key between the two graphs
    "feature", "feature_in_causality_graph",
    # causality-graph only
    "lag", "coefficient", "direction",
    # event-graph only
    "event_summary", "evidence_quote", "scope_justification",
    "quote_similarity", "quote_best_match", "judge_verdict", "judge_reasoning",
    "article_source", "article_date", "article_url", "article_body_preview",
]


def merge_event_causality_graphs(causality_df, data, verbose=True):
    """Stack the causality graph and the event graph into a single edge-list DataFrame.

    causality_df : the causality_graph DataFrame (cause, effect, lag, coefficient, direction)
    data         : list returned by unique_article() / load_llm_article_connections()
    verbose      : print how the two graphs overlap

    Every row is one edge of the merged graph, and no row of either input is dropped:

      edge_type == "causality" -> cause --(direction)--> effect, keeping lag / coefficient
      edge_type == "event"     -> feature --(edge_label)--> "event_summary\\n(date)",
                                  keeping the article metadata and the LLM/judge evidence
                                  (one row per llm_connection, so an article with 3
                                  connections contributes 3 rows)

    The two graphs meet on the feature nodes: `feature_in_causality_graph` says whether an
    event edge hangs off a feature the causality graph actually uses, so
    `merged[merged["feature_in_causality_graph"]]` is the connected part of the merged graph.

    Columns that do not apply to a row are left as None/NaN.
    """
    causality_features = set()
    for column in ("cause", "effect"):
        causality_features.update(
            str(value).strip() for value in causality_df[column].dropna() if str(value).strip()
        )

    rows = []

    # --- causality edges: feature -> feature ---
    for _, edge in causality_df.iterrows():
        cause = edge.get("cause")
        effect = edge.get("effect")
        rows.append(
            {
                "edge_type": "causality",
                "source_node": cause,
                "source_type": "feature",
                "target_node": effect if pd.notna(effect) else None,
                "target_type": "feature" if pd.notna(effect) else None,
                "edge_label": edge.get("direction"),
                "feature": cause,
                "feature_in_causality_graph": True,
                "lag": edge.get("lag"),
                "coefficient": edge.get("coefficient"),
                "direction": edge.get("direction"),
            }
        )

    # --- event edges: feature -> event ---
    for record in data:
        article = record.get("article") or {}
        for connection in record.get("llm_connections") or []:
            feature = connection.get("feature")
            event_summary = connection.get("event_summary")
            verification = connection.get("verification") or {}

            # same node id convention as the event graph builder in notebook 06/07
            event_node = f"{event_summary}\n({article.get('date')})" if event_summary else None

            rows.append(
                {
                    "edge_type": "event",
                    "source_node": feature,
                    "source_type": "feature",
                    "target_node": event_node,
                    "target_type": "event" if event_node else None,
                    "edge_label": connection.get("edge_label"),
                    "feature": feature,
                    "feature_in_causality_graph": str(feature or "").strip() in causality_features,
                    "event_summary": event_summary,
                    "evidence_quote": connection.get("evidence_quote"),
                    "scope_justification": connection.get("scope_justification"),
                    "quote_similarity": verification.get("quote_similarity"),
                    "quote_best_match": verification.get("quote_best_match"),
                    "judge_verdict": verification.get("judge_verdict"),
                    "judge_reasoning": verification.get("judge_reasoning"),
                    "article_source": article.get("source"),
                    "article_date": article.get("date"),
                    "article_url": article.get("url"),
                    "article_body_preview": article.get("body_preview"),
                }
            )

    merged = pd.DataFrame(rows, columns=MERGED_COLUMNS)

    if verbose:
        is_event = merged["edge_type"] == "event"
        n_linked = int(merged.loc[is_event, "feature_in_causality_graph"].sum())
        shared = sorted(
            set(merged.loc[is_event & merged["feature_in_causality_graph"], "feature"].dropna())
        )
        print(
            f"{len(merged)} edge(s) | {int((~is_event).sum())} causality + {int(is_event.sum())} event "
            f"| {n_linked} event edge(s) anchored on a causality feature"
        )
        print(f"shared feature node(s): {', '.join(shared) if shared else 'none'}")

    return merged

In [25]:
merged_graph = merge_event_causality_graphs(causality_graph, unique_articles)

merged_graph

576 edge(s) | 108 causality + 468 event | 98 event edge(s) anchored on a causality feature
shared feature node(s): exports, stock_market_close


,edge_type,source_node,source_type,target_node,target_type,edge_label,feature,feature_in_causality_graph,lag,coefficient,...,evidence_quote,scope_justification,quote_similarity,quote_best_match,judge_verdict,judge_reasoning,article_source,article_date,article_url,article_body_preview
0,causality,Sales_lag_28,feature,Production,feature,positive,Sales_lag_28,True,0.0,0.932276,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,causality,delivery_sales_ratio,feature,production_sales_ratio,feature,positive,delivery_sales_ratio,True,0.0,0.909791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,causality,Sales_lag_28,feature,Sales_lag_3,feature,positive,Sales_lag_28,True,0.0,0.850840,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,causality,Sales_lag_28,feature,Sales_lag_7,feature,positive,Sales_lag_28,True,0.0,0.834155,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,causality,Sales_lag_7,feature,Sales_lag_21,feature,positive,Sales_lag_7,True,0.0,0.747857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,event,inflation_rate,feature,Crises raise prices\n(2022-12-28),event,raises price levels,inflation_rate,False,NaN,NaN,...,The Covid-19 pandemic and the Russia-Ukraine c...,The article attributes broad increases across ...,0.993,"""The Covid-19 pandemic and the Russia-Ukraine ...",CONFIRMED,The article explicitly states that the Covid-1...,MSN International Edition,2022-12-28,https://www.msn.com/en-xl/asia/bangladesh/priv...,"Dhaka, Dec. 28 -- FBCCI president Md Jashim Ud..."
572,event,inflation_rate,feature,Proposed budget targets inflation\n(2023-06-02),event,aims to contain,inflation_rate,False,NaN,NaN,...,mainly aimed to contain inflation and maintain...,The article links the national budget to conta...,0.993,mainly aimed to contain inflation and maintain...,CONFIRMED,The article explicitly states the budget is 'm...,United News of India,2023-06-02,http://www.uniindia.com/news/world/b-desh-fina...,"Dhaka, June 2 (UNI) Bangladesh Finance Ministe..."
573,event,inflation_rate,feature,Consumer prices raise inflation\n(2023-04-05),event,raises inflation,inflation_rate,False,NaN,NaN,...,Inflation in Bangladesh jumped to 9.33 percent...,The article explicitly describes a country-wid...,0.995,Inflation in Bangladesh jumped to 9.33 percent...,CONFIRMED,The article states that Bangladesh’s inflation...,english.news.cn,2023-04-05,https://english.news.cn/asiapacific/20230405/a...,"DHAKA, April 5 (Xinhua) -- Inflation in Bangla..."
574,event,exports,feature,"Policy raises exports, aids recovery\n(2022-09...",event,supports economic recovery,exports,True,NaN,NaN,...,the government's policy of raising exports and...,The article explicitly describes a national go...,0.995,the government's policy of raising exports and...,CONFIRMED,"The article states, “the government's policy o...",LatestLY,2022-09-10,https://www.latestly.com/agency-news/world-new...,"Dhaka [Bangladesh], September 10 (ANI): Econom..."


In [38]:
merged_graph[
    (merged_graph["edge_type"] == "causality")
    & (
        merged_graph["source_node"].isin(existing_macro)
        | merged_graph["target_node"].isin(existing_macro)
    )
]

,edge_type,source_node,source_type,target_node,target_type,edge_label,feature,feature_in_causality_graph,lag,coefficient,...,evidence_quote,scope_justification,quote_similarity,quote_best_match,judge_verdict,judge_reasoning,article_source,article_date,article_url,article_body_preview
8,causality,Temperature,feature,stock_market_close,feature,positive,Temperature,True,0.0,0.590163,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,causality,Temperature,feature,stock_market_close,feature,negative,Temperature,True,1.0,-0.580860,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37,causality,stock_market_close,feature,Sales_lag_2,feature,negative,stock_market_close,True,0.0,-0.186389,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,causality,production_sales_ratio,feature,stock_market_close,feature,negative,production_sales_ratio,True,1.0,-0.103674,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,causality,stock_market_close,feature,Sales_lag_1,feature,positive,stock_market_close,True,1.0,0.089111,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95,causality,delivery_sales_ratio,feature,stock_market_close,feature,positive,delivery_sales_ratio,True,1.0,0.068625,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,causality,stock_market_close,feature,Sales_lag_3,feature,negative,stock_market_close,True,0.0,-0.063023,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,causality,exports,feature,NaN,NaN,none,exports,True,NaN,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
merged_graph.to_csv(f"../../merged_graph/merged_graph_{end_date}_feature_importance_top_{top_n}.csv")

In [34]:
merged_graph[(merged_graph["source_node"] == "Sales") | (merged_graph["target_node"] == "Sales")]

,edge_type,source_node,source_type,target_node,target_type,edge_label,feature,feature_in_causality_graph,lag,coefficient,...,evidence_quote,scope_justification,quote_similarity,quote_best_match,judge_verdict,judge_reasoning,article_source,article_date,article_url,article_body_preview
14,causality,Sales_lag_7,feature,Sales,feature,positive,Sales_lag_7,True,0.0,0.468341,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,causality,Sales_lag_3,feature,Sales,feature,positive,Sales_lag_3,True,0.0,0.239004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,causality,Sales,feature,Sales_lag_3,feature,negative,Sales,True,1.0,-0.232072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50,causality,Sales,feature,inventory_proxy,feature,negative,Sales,True,0.0,-0.162915,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,causality,Sales_lag_28,feature,Sales,feature,positive,Sales_lag_28,True,0.0,0.155784,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,causality,Factory_Issue,feature,Sales,feature,negative,Factory_Issue,True,0.0,-0.145017,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,causality,Sales_lag_21,feature,Sales,feature,positive,Sales_lag_21,True,0.0,0.120675,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71,causality,Sales_lag_2,feature,Sales,feature,positive,Sales_lag_2,True,0.0,0.119887,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,causality,Production,feature,Sales,feature,positive,Production,True,0.0,0.087238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,causality,Product_SOS001L12P,feature,Sales,feature,negative,Product_SOS001L12P,True,0.0,-0.084792,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
from datetime import timedelta
from IPython.display import IFrame

_MERGED_GRAPH_TEMPLATE = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>__TITLE__</title>
<script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
<style>
  html, body { margin: 0; padding: 0; height: 100%; width: 100%; overflow: hidden;
               background: #10141c; color: #e8ecf4;
               font-family: "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
  #graph { position: absolute; top: 0; left: 0; right: 0; bottom: 0; }
  .panel { position: fixed; z-index: 10; background: rgba(22, 28, 40, 0.94);
           border: 1px solid #2c3648; border-radius: 10px; padding: 12px 14px;
           box-shadow: 0 6px 20px rgba(0, 0, 0, 0.45); font-size: 13px; }
  #controls { top: 14px; left: 14px; width: 340px; }
  #legend { bottom: 14px; left: 14px; }
  #controls h3 { margin: 0 0 4px 0; font-size: 15px; font-weight: 600; }
  .sub { color: #8fa0bb; font-size: 11px; margin-bottom: 10px; }
  .row { display: flex; align-items: center; gap: 8px; margin-top: 8px; }
  input[type="date"] { background: #0d1119; color: #e8ecf4; border: 1px solid #38455c;
                       border-radius: 6px; padding: 5px 7px; font-size: 13px; flex: 1; }
  input[type="range"] { width: 100%; accent-color: #ff8f3f; }
  button { background: #232d40; color: #e8ecf4; border: 1px solid #38455c; border-radius: 6px;
           padding: 5px 10px; font-size: 12px; cursor: pointer; }
  button:hover { background: #2f3d55; }
  label.check { display: flex; align-items: center; gap: 6px; color: #b9c6da;
                font-size: 12px; cursor: pointer; }
  #status { margin-top: 10px; padding-top: 9px; border-top: 1px solid #2c3648;
            color: #8fa0bb; font-size: 11.5px; line-height: 1.55; }
  #status b { color: #e8ecf4; font-weight: 600; }
  .key { display: flex; align-items: center; gap: 8px; margin: 5px 0; color: #b9c6da; font-size: 12px; }
  .swatch { width: 15px; height: 15px; border-radius: 50%; flex: none; }
  .swatch.box { border-radius: 3px; }
  .line { width: 24px; height: 0; border-top: 3px solid; flex: none; }
  .line.dash { border-top-style: dashed; }
</style>
</head>
<body>
<div id="graph"></div>

<div class="panel" id="controls">
  <h3>__TITLE__</h3>
  <div class="sub">target variable: <b>__TARGET__</b></div>

  <div class="row">
    <input type="date" id="dateBox" min="__MIN_DATE__" max="__MAX_DATE__" value="__MIN_DATE__">
    <button id="prevBtn" title="previous day">&#9664;</button>
    <button id="nextBtn" title="next day">&#9654;</button>
  </div>
  <div class="row"><input type="range" id="dateSlider" min="0" max="0" value="0" step="1"></div>
  <div class="row">
    <label class="check"><input type="checkbox" id="allDates" checked> all dates</label>
    <label class="check"><input type="checkbox" id="cumulative"> up to this day</label>
  </div>
  <div class="row">
    <button id="fitBtn">fit view</button>
    <button id="focusBtn">focus target</button>
    <button id="playBtn">&#9654; play</button>
  </div>
  <div id="status"></div>
</div>

<div class="panel" id="legend">
  <div class="key"><span class="swatch" style="background:#ff8f3f"></span> target variable</div>
  <div class="key"><span class="swatch" style="background:#4ea4e0"></span> feature (causality graph)</div>
  <div class="key"><span class="swatch" style="background:#57c7ae"></span> feature (event graph only)</div>
  <div class="key"><span class="swatch box" style="background:#b39ddb"></span> event (from article)</div>
  <div class="key"><span class="line" style="border-color:#3fa96b"></span> causal link (positive)</div>
  <div class="key"><span class="line" style="border-color:#e05c5c"></span> causal link (negative)</div>
  <div class="key"><span class="line dash" style="border-color:#a487e8"></span> article evidence</div>
</div>

<script>
const DATA = __DATA__;

const nodes = new vis.DataSet(DATA.nodes);
const edges = new vis.DataSet(DATA.edges);

const state = { allDates: true, cumulative: false, day: DATA.days[0] };

function dayVisible(day) {
  if (state.allDates) return true;
  if (!day) return false;
  return state.cumulative ? day <= state.day : day === state.day;
}
function nodeVisible(node) { return node.group === "feature" ? true : dayVisible(node.day); }
function edgeVisible(edge) { return edge.group === "causality" ? true : dayVisible(edge.day); }

const nodesView = new vis.DataView(nodes, { filter: nodeVisible });
const edgesView = new vis.DataView(edges, { filter: edgeVisible });

const options = {
  physics: { solver: "forceAtlas2Based",
             forceAtlas2Based: { gravitationalConstant: -62, centralGravity: 0.011,
                                 springLength: 130, springConstant: 0.09, avoidOverlap: 0.4 },
             stabilization: { iterations: 220 } },
  interaction: { hover: true, tooltipDelay: 120, navigationButtons: true, keyboard: false,
                 multiselect: true },
  nodes: { font: { color: "#e8ecf4", size: 14, face: "Segoe UI", strokeWidth: 3, strokeColor: "#10141c" } },
  edges: { font: { color: "#9fb0c8", size: 11, face: "Segoe UI", strokeWidth: 4,
                   strokeColor: "#10141c", align: "middle" },
           smooth: { type: "continuous" }, arrows: { to: { enabled: true, scaleFactor: 0.55 } } }
};

const network = new vis.Network(document.getElementById("graph"),
                                { nodes: nodesView, edges: edgesView }, options);

const dateBox = document.getElementById("dateBox");
const slider = document.getElementById("dateSlider");
const allDates = document.getElementById("allDates");
const cumulative = document.getElementById("cumulative");
const status = document.getElementById("status");
const playBtn = document.getElementById("playBtn");

slider.max = String(Math.max(DATA.days.length - 1, 0));

function refresh() {
  nodesView.refresh();
  edgesView.refresh();
  const shownEvents = nodesView.get().filter(n => n.group === "event").length;
  const shownEdges = edgesView.get().filter(e => e.group === "event").length;
  const scope = state.allDates
      ? "all days (" + DATA.days[0] + " to " + DATA.days[DATA.days.length - 1] + ")"
      : (state.cumulative ? "up to " + state.day : state.day);
  status.innerHTML =
      "showing <b>" + scope + "</b><br>" +
      "<b>" + shownEvents + "</b> event node(s), <b>" + shownEdges + "</b> article edge(s)<br>" +
      "<b>" + DATA.causality_edges + "</b> causal edge(s) (always shown)";
}

function setDay(day, syncBox, syncSlider) {
  state.day = day;
  if (syncBox) dateBox.value = day;
  if (syncSlider) {
    const i = DATA.days.indexOf(day);
    if (i >= 0) slider.value = String(i);
  }
  refresh();
}

dateBox.addEventListener("change", () => {
  if (!dateBox.value) return;
  allDates.checked = false; state.allDates = false;
  setDay(dateBox.value, false, true);
});
slider.addEventListener("input", () => {
  allDates.checked = false; state.allDates = false;
  setDay(DATA.days[Number(slider.value)], true, false);
});
document.getElementById("prevBtn").addEventListener("click", () => step(-1));
document.getElementById("nextBtn").addEventListener("click", () => step(1));
function step(delta) {
  allDates.checked = false; state.allDates = false;
  let i = DATA.days.indexOf(state.day);
  if (i < 0) i = 0;
  i = Math.min(Math.max(i + delta, 0), DATA.days.length - 1);
  setDay(DATA.days[i], true, true);
}
allDates.addEventListener("change", () => { state.allDates = allDates.checked; refresh(); });
cumulative.addEventListener("change", () => { state.cumulative = cumulative.checked; refresh(); });
document.getElementById("fitBtn").addEventListener("click", () => network.fit({ animation: true }));
document.getElementById("focusBtn").addEventListener("click", () => {
  if (nodes.get(DATA.target)) network.focus(DATA.target, { scale: 1.1, animation: true });
});

let timer = null;
playBtn.addEventListener("click", () => {
  if (timer) { clearInterval(timer); timer = null; playBtn.innerHTML = "&#9654; play"; return; }
  allDates.checked = false; state.allDates = false;
  playBtn.innerHTML = "&#10074;&#10074; pause";
  timer = setInterval(() => {
    const i = DATA.days.indexOf(state.day);
    if (i >= DATA.days.length - 1) { clearInterval(timer); timer = null; playBtn.innerHTML = "&#9654; play"; return; }
    setDay(DATA.days[i + 1], true, true);
  }, 550);
});

setDay(DATA.days[0], true, true);
</script>
</body>
</html>
"""


def visualize_merged_graph(
    target_variable,
    merged_df,
    end_date,
    top_n,
    output_folder="../../merged_graph",
    title="Merged causality + event graph",
    verbose=True,
):
    """Write an interactive HTML view of the merged graph and return it for the notebook.

    target_variable : name of the target feature node to highlight (e.g. "Sales")
    merged_df       : DataFrame returned by merge_event_causality_graphs()
    end_date, top_n : used for the file name
    output_folder   : folder the html is written to (created if missing)

    The page fills the whole browser window and carries a date control: a date text box, a
    day-by-day slider, prev/next buttons and a play button. Moving through the days filters
    the event graph down to that day, while the causality graph stays on screen as the
    backbone (its edges carry no date). "up to this day" makes the filter cumulative.

    Causality and event parts stay distinguishable:
      features      -> blue dots (teal when the feature is only in the event graph)
      target        -> orange star
      events        -> purple boxes
      causal links  -> solid green / red edges, thickness scaled by |coefficient|
      article links -> dashed purple edges

    Returns an IFrame of the saved page.
    """
    events = merged_df[merged_df["edge_type"] == "event"]
    causal = merged_df[merged_df["edge_type"] == "causality"]

    target = str(target_variable).strip()

    def _text(value):
        return "" if value is None or (isinstance(value, float) and pd.isna(value)) else str(value)

    # ---------------- nodes ----------------
    node_map = {}

    def _feature_node(name, in_causality):
        name = _text(name).strip()
        if not name:
            return None
        node = node_map.get(name)
        if node is None:
            is_target = name.casefold() == target.casefold()
            node = {
                "id": name,
                "label": name,
                "group": "feature",
                "shape": "star" if is_target else "dot",
                "size": 42 if is_target else 26,
                "color": {"background": "#ff8f3f" if is_target else ("#4ea4e0" if in_causality else "#57c7ae"),
                          "border": "#0d1119",
                          "highlight": {"background": "#ffc38a", "border": "#ffffff"}},
                "font": {"size": 20 if is_target else 15},
                "title": ("TARGET VARIABLE\n" if is_target else "")
                         + f"feature: {name}\nin causality graph: {'yes' if in_causality else 'no'}",
            }
            node_map[name] = node
        elif in_causality and node["color"]["background"] == "#57c7ae":
            node["color"]["background"] = "#4ea4e0"
            node["title"] = f"feature: {name}\nin causality graph: yes"
        return node["id"]

    for _, row in causal.iterrows():
        _feature_node(row["source_node"], True)
        _feature_node(row["target_node"], True)

    for _, row in events.iterrows():
        _feature_node(row["feature"], bool(row["feature_in_causality_graph"]))

        event_id = _text(row["target_node"])
        if not event_id:
            continue
        day = _text(row["article_date"])[:10]
        detail = (
            f"{_text(row['event_summary'])}\n"
            f"{_text(row['article_source'])} | {day}\n"
            f"{_text(row['feature'])} --{_text(row['edge_label'])}-->\n"
            f'quote: "{_text(row["evidence_quote"])}"\n'
            f"judge: {_text(row['judge_verdict'])} (similarity {_text(row['quote_similarity'])})\n"
            f"{_text(row['article_url'])}"
        )
        node = node_map.get(event_id)
        if node is None:
            node_map[event_id] = {
                "id": event_id,
                "label": _text(row["event_summary"]),
                "group": "event",
                "day": day,
                "shape": "box",
                "color": {"background": "#b39ddb", "border": "#0d1119",
                          "highlight": {"background": "#d7c7f5", "border": "#ffffff"}},
                "font": {"size": 13, "color": "#161c28", "strokeWidth": 0},
                "margin": 8,
                "title": detail,
            }
        elif detail not in node["title"]:
            node["title"] += "\n\n" + detail

    # ---------------- edges ----------------
    edge_map = {}

    for _, row in causal.iterrows():
        source, target_node = _text(row["source_node"]), _text(row["target_node"])
        if not source or not target_node:
            continue  # isolated causality row (no effect) -> the node is kept, no edge
        direction = _text(row["direction"]) or "none"
        coefficient = row["coefficient"]
        coefficient = 0.0 if pd.isna(coefficient) else float(coefficient)
        lag = "" if pd.isna(row["lag"]) else f"{float(row['lag']):g}"
        key = ("causality", source, target_node, direction, lag)
        if key in edge_map:
            continue
        color = {"positive": "#3fa96b", "negative": "#e05c5c"}.get(direction, "#8a97ab")
        edge_map[key] = {
            "id": f"c{len(edge_map)}",
            "from": source, "to": target_node,
            "group": "causality",
            "label": f"{coefficient:+.2f}",
            "width": round(1.2 + 4.5 * abs(coefficient), 2),
            "color": {"color": color, "highlight": "#ffffff", "opacity": 0.85},
            "title": f"causal link\n{source} -> {target_node}\ndirection: {direction}\n"
                     f"coefficient: {coefficient:.4f}\nlag: {lag}",
        }

    for _, row in events.iterrows():
        source, target_node = _text(row["feature"]).strip(), _text(row["target_node"])
        if not source or not target_node:
            continue
        label = _text(row["edge_label"])
        day = _text(row["article_date"])[:10]
        key = ("event", source, target_node, label, day)
        if key in edge_map:
            continue
        edge_map[key] = {
            "id": f"e{len(edge_map)}",
            "from": source, "to": target_node,
            "group": "event",
            "day": day,
            "label": label,
            "width": 1.6,
            "dashes": [6, 5],
            "color": {"color": "#a487e8", "highlight": "#ffffff", "opacity": 0.8},
            "title": f"article evidence ({day})\n{source} --{label}--> {_text(row['event_summary'])}\n"
                     f"source: {_text(row['article_source'])}\njudge: {_text(row['judge_verdict'])}",
        }

    # ------- day axis: every calendar day between the first and the last article -------
    days_with_events = sorted({_text(d)[:10] for d in events["article_date"].dropna() if _text(d)})
    if days_with_events:
        cursor, last = _to_date(days_with_events[0]), _to_date(days_with_events[-1])
        days = []
        while cursor <= last:
            days.append(cursor.isoformat())
            cursor += timedelta(days=1)
    else:
        days = [_text(end_date)[:10]]

    payload = {
        "nodes": list(node_map.values()),
        "edges": list(edge_map.values()),
        "days": days,
        "target": target,
        "causality_edges": sum(1 for e in edge_map.values() if e["group"] == "causality"),
    }

    html = (
        _MERGED_GRAPH_TEMPLATE
        .replace("__DATA__", json.dumps(payload, ensure_ascii=False).replace("</", "<\\/"))
        .replace("__TITLE__", title)
        .replace("__TARGET__", target)
        .replace("__MIN_DATE__", days[0])
        .replace("__MAX_DATE__", days[-1])
    )

    folder = Path(output_folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / f"merged_graph_{end_date}_feature_importance_top_{top_n}.html"
    output_path.write_text(html, encoding="utf-8")

    if verbose:
        n_events = sum(1 for n in node_map.values() if n["group"] == "event")
        print(
            f"{len(node_map)} node(s) ({len(node_map) - n_events} feature, {n_events} event) "
            f"| {len(edge_map)} edge(s) ({payload['causality_edges']} causal, "
            f"{len(edge_map) - payload['causality_edges']} article) "
            f"| {len(days)} day(s) on the slider"
        )
        print(f"saved to {output_path.resolve()}")

    return IFrame(str(output_path), width="100%", height=820)

In [36]:
visualize_merged_graph("Sales", merged_graph, end_date=end_date, top_n=top_n)

439 node(s) (25 feature, 414 event) | 540 edge(s) (107 causal, 433 article) | 330 day(s) on the slider
saved to C:\Users\Iman\OneDrive\Desktop\Supply Chain Study\merged_graph\merged_graph_2023-07-17_feature_importance_top_20.html
